# Solving the Butler--Volmer equation

We now move from equilibrium double-layer structure to **charge-transfer kinetics**.  
The Butler--Volmer equation describes the faradaic current density generated when an electrode is moved away from equilibrium by an overpotential

$$
\eta = E - E_{\mathrm{eq}} .
$$

For an elementary charge-transfer reaction

$$
\mathrm{Ox} + n e^- \rightleftharpoons \mathrm{Red},
$$

the net current is written as the sum of an anodic oxidation branch and a cathodic reduction branch:

$$
\boxed{
 i(\eta)
 = i_0\left[
 \exp\left(\frac{(1-\alpha)nF\eta}{RT}\right)
 -
 \exp\left(-\frac{\alpha nF\eta}{RT}\right)
 \right]
}
$$

where $i_0$ is the exchange current density, $\alpha$ is the cathodic transfer coefficient, $n$ is the number of transferred electrons, $T$ is the temperature, $F$ is the Faraday constant, and $R$ is the gas constant.

# Parametric setup

The following code cell defines the physical constants, unit conversions, and the parameter container used by the interactive Butler--Volmer model.

The parameters are:

| Symbol | Unit | Meaning |
|---|---:|---|
| $\eta$ | V | overpotential, i.e. deviation from equilibrium |
| $i_0$ | A m$^{-2}$ | exchange current density |
| $\alpha$ | -- | cathodic charge-transfer coefficient |
| $1-\alpha$ | -- | anodic charge-transfer coefficient |
| $n$ | -- | number of transferred electrons |
| $T$ | K | temperature |
| $i$ | A m$^{-2}$ | net current density |

In [ ]:
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAVE_IPYWIDGETS = True
except Exception:
    HAVE_IPYWIDGETS = False


# Fundamental constants
F = 96485.33212       # C mol^-1
R = 8.314462618      # J mol^-1 K^-1

# Unit conversion
A_PER_M2_TO_MA_PER_CM2 = 0.1


@dataclass
class BVParams:
    i0: float = 1.0       # A m^-2
    alpha: float = 0.5    # cathodic transfer coefficient
    n: float = 1.0        # number of transferred electrons
    T: float = 298.15     # K


def validate_params(p: BVParams):
    if p.i0 <= 0:
        raise ValueError("The exchange current density i0 must be positive.")
    if not (0.0 < p.alpha < 1.0):
        raise ValueError("The transfer coefficient alpha must satisfy 0 < alpha < 1.")
    if p.n <= 0:
        raise ValueError("The number of transferred electrons n must be positive.")
    if p.T <= 0:
        raise ValueError("The temperature T must be positive.")

# Butler--Volmer current branches

The Butler--Volmer equation is built from two exponential contributions:

$$
 i_a(\eta)=i_0\exp\left(\frac{(1-\alpha)nF\eta}{RT}\right),
 \qquad
 i_c(\eta)=-i_0\exp\left(-\frac{\alpha nF\eta}{RT}\right).
$$

The net current density is therefore

$$
 i(\eta)=i_a(\eta)+i_c(\eta).
$$

At equilibrium, $\eta=0$, the two branches have equal magnitude and opposite sign:

$$
 i_a(0)=i_0,
 \qquad
 i_c(0)=-i_0,
 \qquad
 i(0)=0.
$$

Thus, $i_0$ is not a net current. It is the magnitude of the forward and backward charge-transfer currents at equilibrium.

In [ ]:
def i_anodic(eta, p: BVParams):
    validate_params(p)
    eta = np.asarray(eta, dtype=float)
    exponent = (1.0 - p.alpha) * p.n * F * eta / (R * p.T)
    return p.i0 * np.exp(np.clip(exponent, -700.0, 700.0))


def i_cathodic(eta, p: BVParams):
    validate_params(p)
    eta = np.asarray(eta, dtype=float)
    exponent = -p.alpha * p.n * F * eta / (R * p.T)
    return -p.i0 * np.exp(np.clip(exponent, -700.0, 700.0))


def i_butler_volmer(eta, p: BVParams):
    return i_anodic(eta, p) + i_cathodic(eta, p)


def i_butler_volmer_linearized(eta, p: BVParams):
    """
    Small-overpotential limit of Butler--Volmer.

    For eta -> 0,

        i(eta) ~ i0 * nF eta / RT.

    This is the charge-transfer-resistance / micropolarization limit.
    It is independent of alpha to first order.
    """
    validate_params(p)
    eta = np.asarray(eta, dtype=float)
    return p.i0 * p.n * F * eta / (R * p.T)


def log10_abs_current_for_tafel_plot(eta, p: BVParams, eta_local=1e-3):
    """
    Compute log10(|i|) for the Tafel representation.

    Close to eta = 0, the full Butler--Volmer current is replaced by its
    local asymptotic expansion,

        i(eta) ~ i0 * nF eta / RT.

    This avoids numerical cancellation near equilibrium. The point eta = 0
    itself is still undefined because log10(0) = -infinity, so it is masked.
    """
    validate_params(p)
    eta = np.asarray(eta, dtype=float)

    current = i_butler_volmer(eta, p)
    local_current = i_butler_volmer_linearized(eta, p)

    use_local = np.abs(eta) <= eta_local
    current = np.where(use_local, local_current, current)

    current_mA = A_PER_M2_TO_MA_PER_CM2 * current

    y = np.full_like(eta, np.nan, dtype=float)
    mask = np.abs(current_mA) > 0.0
    y[mask] = np.log10(np.abs(current_mA[mask]))

    return y


def tafel_anodic(eta, p: BVParams):
    return i_anodic(eta, p)


def tafel_cathodic(eta, p: BVParams):
    return i_cathodic(eta, p)


def tafel_slope_anodic_base10(p: BVParams):
    validate_params(p)
    return 2.303 * R * p.T / ((1.0 - p.alpha) * p.n * F)


def tafel_slope_cathodic_base10(p: BVParams):
    validate_params(p)
    return -2.303 * R * p.T / (p.alpha * p.n * F)


def charge_transfer_resistance_area(p: BVParams):
    validate_params(p)
    return R * p.T / (p.n * F * p.i0)


### Local behavior near equilibrium

The Tafel representation uses the logarithm of the absolute net current density,

$$
\log_{10}|i|.
$$

This transformation is useful at sufficiently large positive or negative overpotential, where one Butler--Volmer exponential dominates and the response becomes approximately linear. Near equilibrium, however, the net current behaves differently. At equilibrium,

$$
\eta = 0,
$$

the anodic and cathodic currents have equal magnitude and opposite sign,

$$
i_a(0)=i_0,
\qquad
i_c(0)=-i_0.
$$

Therefore, the net current vanishes,

$$
i(0)=i_a(0)+i_c(0)=0.
$$

The logarithmic quantity is then not defined,

$$
\log_{10}|i(0)|=\log_{10}(0).
$$

This is not a removable numerical artifact. It appears because the logarithm is applied to a physical current that goes exactly to zero at equilibrium. The point $\eta=0$ must therefore remain excluded in the Tafel representation.

To evaluate the neighboring points in a stable way, we use the local Butler--Volmer expansion. Starting from

$$
i(\eta)
=
i_0
\left[
\exp\left(\frac{(1-\alpha)nF\eta}{RT}\right)
-
\exp\left(-\frac{\alpha nF\eta}{RT}\right)
\right],
$$

the small-overpotential limit gives

$$
i(\eta)
\sim
i_0\frac{nF}{RT}\eta,
\qquad
\eta\to 0.
$$

Thus, close to equilibrium,

$$
\log_{10}|i|
\sim
\log_{10}
\left(
i_0\frac{nF}{RT}|\eta|
\right).
$$

In the code, a small internal threshold is preset around $\eta=0$. Inside this threshold, the logarithmic Tafel quantity is evaluated from the local expansion instead of from the direct Butler--Volmer expression. This avoids unstable cancellation between the anodic and cathodic exponentials when the net current is very small. The exact point $\eta=0$ is still excluded, because the net current is exactly zero there and $\log_{10}(0)$ is undefined. The local expansion therefore does not remove the singularity; it only gives a stable and physically meaningful description of the neighboring points.

# Helper functions

The next functions support the interactive sweep. They parse comma-separated sweep values, create a parameter set for each curve, generate labels, and keep the plots visually consistent with the previous notebooks.

The same logic is used as in the Gouy--Chapman, Gouy--Chapman--Stern, Bikerman, and Freise notebooks: one input is swept while all other parameters are kept fixed at their base values.

In [ ]:
def parse_sweep_values(text):
    values = []
    for part in text.replace(";", ",").split(","):
        part = part.strip()
        if part:
            values.append(float(part))
    if not values:
        raise ValueError("Please provide at least one sweep value.")
    return values


def make_params(input_name, value, base):
    p = BVParams(
        i0=base.i0,
        alpha=base.alpha,
        n=base.n,
        T=base.T,
    )

    if input_name == "Exchange current density":
        p.i0 = value
    elif input_name == "Transfer coefficient":
        p.alpha = value
    elif input_name == "Number of electrons":
        p.n = value
    elif input_name == "Temperature":
        p.T = value
    else:
        raise ValueError(f"Unknown input name: {input_name}")

    validate_params(p)
    return p


def label_for_value(input_name, value):
    if input_name == "Exchange current density":
        return rf"$i_0={value:g}$ A m$^{{-2}}$"
    if input_name == "Transfer coefficient":
        return rf"$\alpha={value:g}$"
    if input_name == "Number of electrons":
        return rf"$n={value:g}$"
    if input_name == "Temperature":
        return rf"$T={value:g}$ K"
    return f"{input_name}={value:g}"


def color_sweep(color_name, n):
    maps = {
        "black": plt.cm.Greys,
        "red": plt.cm.Reds,
        "blue": plt.cm.Blues,
        "green": plt.cm.Greens,
        "purple": plt.cm.Purples,
    }
    cmap = maps[color_name]
    return cmap(np.linspace(0.40, 0.88, max(n, 2)))[:n]


def add_zero_axes(ax):
    ax.axhline(0.0, color="0.35", lw=1.0)
    ax.axvline(0.0, color="0.35", lw=1.0)


def clean_axis(ax):
    ax.grid(True, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)

# From Butler--Volmer to Tafel behavior

For large positive overpotential, the anodic exponential dominates:

$$
\eta \gg 0,
\qquad
 i(\eta)\approx i_a(\eta)
 = i_0\exp\left(\frac{(1-\alpha)nF\eta}{RT}\right).
$$

For large negative overpotential, the cathodic exponential dominates:

$$
\eta \ll 0,
\qquad
 i(\eta)\approx i_c(\eta)
 = -i_0\exp\left(-\frac{\alpha nF\eta}{RT}\right).
$$

Using base-10 logarithms, the Tafel slopes are

$$
 b_{a,10}=\frac{2.303RT}{(1-\alpha)nF},
 \qquad
 b_{c,10}=-\frac{2.303RT}{\alpha nF}.
$$

These slopes have units of V per decade. The Tafel plot uses $\log_{10}|i|$, so the region close to $i=0$ must be excluded to avoid the logarithmic singularity at equilibrium.

In [ ]:
def plot_butler_volmer_sweep(
    input_name,
    sweep_values,
    eta_min,
    eta_max,
    npts,
    base_i0,
    base_alpha,
    base_n,
    base_T,
    save_path=None,
    dpi=300,
):
    if eta_min >= eta_max:
        raise ValueError("eta_min must be smaller than eta_max.")
    if npts < 10:
        raise ValueError("npts must be at least 10.")

    # Internal plotting constants.
    # These are not exposed to the student interface.
    tafel_current_cutoff = 1e-12
    tafel_eta_cutoff = 1e-3

    base = BVParams(
        i0=base_i0,
        alpha=base_alpha,
        n=base_n,
        T=base_T,
    )
    validate_params(base)

    eta = np.linspace(eta_min, eta_max, int(npts))
    sweep_values = [float(v) for v in sweep_values]
    ncurves = len(sweep_values)

    potential_colors = color_sweep("black", ncurves)
    anodic_colors = color_sweep("red", ncurves)
    cathodic_colors = color_sweep("blue", ncurves)
    tafel_colors = color_sweep("blue", ncurves)
    comparison_colors = color_sweep("purple", ncurves)

    title_fontsize = 12
    label_fontsize = 11
    tick_fontsize = 10
    legend_fontsize = 7.5

    fig = plt.figure(figsize=(13.2, 7.4), constrained_layout=True)
    gs = fig.add_gridspec(2, 6)

    ax_potential = fig.add_subplot(gs[0, 0:2])
    ax_anodic = fig.add_subplot(gs[0, 2:4])
    ax_cathodic = fig.add_subplot(gs[0, 4:6])

    ax_tafel = fig.add_subplot(gs[1, 0:3])
    ax_comparison = fig.add_subplot(gs[1, 3:6])

    summary = []

    for j, value in enumerate(sweep_values):
        p = make_params(input_name, value, base)
        label = label_for_value(input_name, value)

        ia = i_anodic(eta, p)
        ic = i_cathodic(eta, p)
        inet = ia + ic

        ia_mA = A_PER_M2_TO_MA_PER_CM2 * ia
        ic_mA = A_PER_M2_TO_MA_PER_CM2 * ic
        inet_mA = A_PER_M2_TO_MA_PER_CM2 * inet

        ax_potential.plot(
            eta,
            inet_mA,
            color=potential_colors[j],
            lw=2.4,
            label=label,
        )

        ax_anodic.plot(
            eta,
            ia_mA,
            color=anodic_colors[j],
            lw=2.2,
            ls="-",
            label=label,
        )

        ax_cathodic.plot(
            eta,
            ic_mA,
            color=cathodic_colors[j],
            lw=2.2,
            ls="-",
            label=label,
        )

        tafel_y = log10_abs_current_for_tafel_plot(
            eta,
            p,
            eta_local=tafel_eta_cutoff,
        )

        abs_i_mA = np.abs(inet_mA)

        mask_tafel = (
            np.isfinite(tafel_y)
            & (abs_i_mA > tafel_current_cutoff)
        )

        ax_tafel.plot(
            eta[mask_tafel],
            tafel_y[mask_tafel],
            color=tafel_colors[j],
            lw=2.4,
            label=label,
        )

        # Conventional Tafel-guide panel:
        # eta is plotted against log10(|i|), so slopes are in V/decade.
        tafel_full_y = log10_abs_current_for_tafel_plot(
            eta,
            p,
            eta_local=tafel_eta_cutoff,
        )
        mask_full = np.isfinite(tafel_full_y)

        ax_comparison.plot(
            tafel_full_y[mask_full],
            eta[mask_full],
            color=comparison_colors[j],
            lw=2.2,
            ls="-",
            label=label + " full BV",
        )

        ia_tafel_mA = A_PER_M2_TO_MA_PER_CM2 * tafel_anodic(eta, p)
        mask_anodic = eta > 0.08
        ax_comparison.plot(
            np.log10(np.abs(ia_tafel_mA[mask_anodic])),
            eta[mask_anodic],
            color=comparison_colors[j],
            lw=1.7,
            ls=":",
            label=label + " anodic Tafel",
        )

        ic_tafel_mA = A_PER_M2_TO_MA_PER_CM2 * tafel_cathodic(eta, p)
        mask_cathodic = eta < -0.08
        ax_comparison.plot(
            np.log10(np.abs(ic_tafel_mA[mask_cathodic])),
            eta[mask_cathodic],
            color=comparison_colors[j],
            lw=1.7,
            ls="--",
            label=label + " cathodic Tafel",
        )

        summary.append(
            (
                label,
                tafel_slope_anodic_base10(p),
                tafel_slope_cathodic_base10(p),
                charge_transfer_resistance_area(p),
            )
        )

    add_zero_axes(ax_potential)
    ax_potential.set_title("Net Butler--Volmer current", fontsize=title_fontsize)
    ax_potential.set_xlabel(r"Overpotential $\eta$ (V)", fontsize=label_fontsize)
    ax_potential.set_ylabel(
        r"Current density $i$ (mA/cm$^2$)",
        fontsize=label_fontsize,
    )
    ax_potential.tick_params(axis="both", labelsize=tick_fontsize)
    ax_potential.legend(fontsize=legend_fontsize, frameon=True)
    clean_axis(ax_potential)

    add_zero_axes(ax_anodic)
    ax_anodic.set_title("Anodic current", fontsize=title_fontsize)
    ax_anodic.set_xlabel(r"Overpotential $\eta$ (V)", fontsize=label_fontsize)
    ax_anodic.set_ylabel(
        r"Anodic current density $i_a$ (mA/cm$^2$)",
        fontsize=label_fontsize,
    )
    ax_anodic.tick_params(axis="both", labelsize=tick_fontsize)
    ax_anodic.legend(fontsize=legend_fontsize, frameon=True)
    clean_axis(ax_anodic)

    add_zero_axes(ax_cathodic)
    ax_cathodic.set_title("Cathodic current", fontsize=title_fontsize)
    ax_cathodic.set_xlabel(r"Overpotential $\eta$ (V)", fontsize=label_fontsize)
    ax_cathodic.set_ylabel(
        r"Cathodic current density $i_c$ (mA/cm$^2$)",
        fontsize=label_fontsize,
    )
    ax_cathodic.tick_params(axis="both", labelsize=tick_fontsize)
    ax_cathodic.legend(fontsize=legend_fontsize, frameon=True)
    clean_axis(ax_cathodic)

    ax_tafel.axvline(0.0, color="0.35", lw=1.0)
    ax_tafel.set_title("Semi-log BV representation", fontsize=title_fontsize)
    ax_tafel.set_xlabel(r"Overpotential $\eta$ (V)", fontsize=label_fontsize)
    ax_tafel.set_ylabel(
        r"$\log_{10}(|i|)$ with $i$ in mA/cm$^2$",
        fontsize=label_fontsize,
    )
    ax_tafel.tick_params(axis="both", labelsize=tick_fontsize)
    ax_tafel.legend(fontsize=legend_fontsize, frameon=True)
    clean_axis(ax_tafel)

    ax_comparison.axhline(0.0, color="0.35", lw=1.0)
    ax_comparison.set_title("Conventional Tafel plot", fontsize=title_fontsize)
    ax_comparison.set_xlabel(
        r"$\log_{10}(|i|)$ with $i$ in mA/cm$^2$",
        fontsize=label_fontsize,
    )
    ax_comparison.set_ylabel(r"Overpotential $\eta$ (V)", fontsize=label_fontsize)
    ax_comparison.tick_params(axis="both", labelsize=tick_fontsize)
    ax_comparison.legend(fontsize=legend_fontsize, frameon=True)
    clean_axis(ax_comparison)

    lines = ["Tafel slopes and charge-transfer resistance:"]
    for label, ba, bc, rct in summary[:4]:
        clean_label = (
            label.replace("$", "")
            .replace("\\alpha", "alpha")
            .replace("^{{-2}}", "^-2")
        )
        lines.append(
            f"{clean_label}: ba={ba:.3g} V/dec, "
            f"bc={bc:.3g} V/dec, RctA={rct:.3g} Ω m²"
        )

    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", facecolor="white")

    return fig

def plot_question1_bv_decomposition_guide_2x3(
    alpha_values=(0.3, 0.5, 0.7),
    i0_values=(0.1, 1.0, 10.0),
    eta_min=-0.30,
    eta_max=0.30,
    npts=500,
    base_i0=1.0,
    base_alpha=0.5,
    base_n=1.0,
    base_T=298.15,
    save_path=None,
    dpi=300,
):
    eta = np.linspace(eta_min, eta_max, int(npts))

    alpha_values = [float(v) for v in alpha_values]
    i0_values = [float(v) for v in i0_values]

    fig = plt.figure(figsize=(13.5, 7.4), constrained_layout=True)
    gs = fig.add_gridspec(2, 3)

    ax11 = fig.add_subplot(gs[0, 0])
    ax12 = fig.add_subplot(gs[0, 1])
    ax13 = fig.add_subplot(gs[0, 2])

    ax21 = fig.add_subplot(gs[1, 0])
    ax22 = fig.add_subplot(gs[1, 1])
    ax23 = fig.add_subplot(gs[1, 2])

    alpha_net_colors = color_sweep("black", len(alpha_values))
    alpha_an_colors = color_sweep("red", len(alpha_values))
    alpha_cat_colors = color_sweep("blue", len(alpha_values))

    i0_net_colors = color_sweep("black", len(i0_values))
    i0_an_colors = color_sweep("red", len(i0_values))
    i0_cat_colors = color_sweep("blue", len(i0_values))

    # Top row: alpha sweep
    for j, alpha in enumerate(alpha_values):
        p = BVParams(i0=base_i0, alpha=alpha, n=base_n, T=base_T)
        label = rf"$\alpha = {alpha:g}$"

        ia = A_PER_M2_TO_MA_PER_CM2 * i_anodic(eta, p)
        ic = A_PER_M2_TO_MA_PER_CM2 * i_cathodic(eta, p)
        inet = ia + ic

        ax11.plot(eta, inet, color=alpha_net_colors[j], lw=2.3, label=label)
        ax12.plot(eta, ia, color=alpha_an_colors[j], lw=2.3, label=label)
        ax13.plot(eta, ic, color=alpha_cat_colors[j], lw=2.3, label=label)

    # Bottom row: i0 sweep
    for j, i0 in enumerate(i0_values):
        p = BVParams(i0=i0, alpha=base_alpha, n=base_n, T=base_T)
        label = rf"$i_0 = {i0:g}\ \mathrm{{A/m^2}}$"

        ia = A_PER_M2_TO_MA_PER_CM2 * i_anodic(eta, p)
        ic = A_PER_M2_TO_MA_PER_CM2 * i_cathodic(eta, p)
        inet = ia + ic

        ax21.plot(eta, inet, color=i0_net_colors[j], lw=2.3, label=label)
        ax22.plot(eta, ia, color=i0_an_colors[j], lw=2.3, label=label)
        ax23.plot(eta, ic, color=i0_cat_colors[j], lw=2.3, label=label)

    # Formatting
    for ax in (ax11, ax12, ax13, ax21, ax22, ax23):
        add_zero_axes(ax)
        ax.set_xlabel(r"Overpotential $\eta$ (V)", fontsize=11)
        ax.tick_params(axis="both", labelsize=10)
        clean_axis(ax)

    ax11.set_title(r"Net BV current ($\alpha$-sweep)", fontsize=12)
    ax12.set_title(r"Anodic current ($\alpha$-sweep)", fontsize=12)
    ax13.set_title(r"Cathodic current ($\alpha$-sweep)", fontsize=12)

    ax21.set_title(r"Net BV current ($i_0$-sweep)", fontsize=12)
    ax22.set_title(r"Anodic current ($i_0$-sweep)", fontsize=12)
    ax23.set_title(r"Cathodic current ($i_0$-sweep)", fontsize=12)

    ax11.set_ylabel(r"Current density $i$ (mA/cm$^2$)", fontsize=11)
    ax12.set_ylabel(r"Anodic current density $i_a$ (mA/cm$^2$)", fontsize=11)
    ax13.set_ylabel(r"Cathodic current density $i_c$ (mA/cm$^2$)", fontsize=11)

    ax21.set_ylabel(r"Current density $i$ (mA/cm$^2$)", fontsize=11)
    ax22.set_ylabel(r"Anodic current density $i_a$ (mA/cm$^2$)", fontsize=11)
    ax23.set_ylabel(r"Cathodic current density $i_c$ (mA/cm$^2$)", fontsize=11)

    ax11.legend(fontsize=8, frameon=True)
    ax12.legend(fontsize=8, frameon=True)
    ax13.legend(fontsize=8, frameon=True)

    ax21.legend(fontsize=8, frameon=True)
    ax22.legend(fontsize=8, frameon=True)
    ax23.legend(fontsize=8, frameon=True)

    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", facecolor="white")

    return fig

def plot_question2_tafel_guide_general(
    input_name="Transfer coefficient",
    sweep_values=(0.3, 0.5, 0.7),
    eta_min=-0.30,
    eta_max=0.30,
    npts=500,
    base_i0=1.0,
    base_alpha=0.5,
    base_n=1.0,
    base_T=298.15,
    tafel_eta_cutoff=1e-3,
    save_path=None,
    dpi=300,
):
    eta = np.linspace(eta_min, eta_max, int(npts))
    sweep_values = [float(v) for v in sweep_values]
    ncurves = len(sweep_values)

    colors = color_sweep("purple", ncurves)

    fig = plt.figure(figsize=(7.4, 4.8), constrained_layout=True)
    ax = fig.add_subplot(111)

    base = BVParams(
        i0=base_i0,
        alpha=base_alpha,
        n=base_n,
        T=base_T,
    )
    validate_params(base)

    for j, value in enumerate(sweep_values):
        p = make_params(input_name, value, base)
        label = label_for_value(input_name, value)

        tafel_full_y = log10_abs_current_for_tafel_plot(
            eta,
            p,
            eta_local=tafel_eta_cutoff,
        )
        mask_full = np.isfinite(tafel_full_y)

        ax.plot(
            tafel_full_y[mask_full],
            eta[mask_full],
            color=colors[j],
            lw=2.2,
            ls="-",
            label=label + " full BV",
        )

        ia_tafel_mA = A_PER_M2_TO_MA_PER_CM2 * tafel_anodic(eta, p)
        mask_anodic = eta > 0.08
        ax.plot(
            np.log10(np.abs(ia_tafel_mA[mask_anodic])),
            eta[mask_anodic],
            color=colors[j],
            lw=1.6,
            ls=":",
            label=label + " anodic Tafel",
        )

        ic_tafel_mA = A_PER_M2_TO_MA_PER_CM2 * tafel_cathodic(eta, p)
        mask_cathodic = eta < -0.08
        ax.plot(
            np.log10(np.abs(ic_tafel_mA[mask_cathodic])),
            eta[mask_cathodic],
            color=colors[j],
            lw=1.6,
            ls="--",
            label=label + " cathodic Tafel",
        )

    ax.axhline(0.0, color="0.35", lw=1.0)
    ax.set_title("Conventional Tafel plot", fontsize=12)
    ax.set_xlabel(r"$\log_{10}(|i|)$ with $i$ in mA/cm$^2$", fontsize=11)
    ax.set_ylabel(r"Overpotential $\eta$ (V)", fontsize=11)
    ax.tick_params(axis="both", labelsize=10)
    ax.legend(fontsize=7, frameon=True)
    clean_axis(ax)

    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", facecolor="white")

    return fig


def plot_question2_tafel_guides_1x3(
    alpha_values=(0.3, 0.5, 0.7),
    n_values=(1, 2, 4),
    T_values=(273.15, 298.15, 323.15),
    eta_min=-0.30,
    eta_max=0.30,
    npts=500,
    base_i0=1.0,
    base_alpha=0.5,
    base_n=1.0,
    base_T=298.15,
    tafel_eta_cutoff=1e-3,
    save_path=None,
    dpi=300,
):
    eta = np.linspace(eta_min, eta_max, int(npts))

    alpha_values = [float(v) for v in alpha_values]
    n_values = [float(v) for v in n_values]
    T_values = [float(v) for v in T_values]

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(13.8, 4.3),
        constrained_layout=True,
    )

    ax_alpha, ax_n, ax_T = axes

    alpha_colors = color_sweep("purple", len(alpha_values))
    n_colors = color_sweep("blue", len(n_values))
    T_colors = color_sweep("red", len(T_values))

    def draw_tafel_panel(ax, param_name, values, colors):
        base = BVParams(
            i0=base_i0,
            alpha=base_alpha,
            n=base_n,
            T=base_T,
        )
        validate_params(base)

        for j, value in enumerate(values):
            p = make_params(param_name, value, base)
            label = label_for_value(param_name, value)

            tafel_full_y = log10_abs_current_for_tafel_plot(
                eta,
                p,
                eta_local=tafel_eta_cutoff,
            )
            mask_full = np.isfinite(tafel_full_y)

            ax.plot(
                tafel_full_y[mask_full],
                eta[mask_full],
                color=colors[j],
                lw=2.2,
                ls="-",
                label=label + " full BV",
            )

            ia_tafel_mA = A_PER_M2_TO_MA_PER_CM2 * tafel_anodic(eta, p)
            mask_anodic = eta > 0.08
            ax.plot(
                np.log10(np.abs(ia_tafel_mA[mask_anodic])),
                eta[mask_anodic],
                color=colors[j],
                lw=1.5,
                ls=":",
                label=label + " anodic Tafel",
            )

            ic_tafel_mA = A_PER_M2_TO_MA_PER_CM2 * tafel_cathodic(eta, p)
            mask_cathodic = eta < -0.08
            ax.plot(
                np.log10(np.abs(ic_tafel_mA[mask_cathodic])),
                eta[mask_cathodic],
                color=colors[j],
                lw=1.5,
                ls="--",
                label=label + " cathodic Tafel",
            )

        ax.axhline(0.0, color="0.35", lw=1.0)
        ax.set_xlabel(r"$\log_{10}(|i|)$ with $i$ in mA/cm$^2$", fontsize=11)
        ax.set_ylabel(r"Overpotential $\eta$ (V)", fontsize=11)
        ax.tick_params(axis="both", labelsize=10)
        ax.legend(fontsize=6.8, frameon=True)
        clean_axis(ax)

    draw_tafel_panel(ax_alpha, "Transfer coefficient", alpha_values, alpha_colors)
    draw_tafel_panel(ax_n, "Number of electrons", n_values, n_colors)
    draw_tafel_panel(ax_T, "Temperature", T_values, T_colors)

    ax_alpha.set_title(r"Tafel plot ($\alpha$-sweep)", fontsize=12)
    ax_n.set_title(r"Tafel plot ($n$-sweep)", fontsize=12)
    ax_T.set_title(r"Tafel plot ($T$-sweep)", fontsize=12)

    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight", facecolor="white")

    return fig

In [ ]:
if HAVE_IPYWIDGETS:
    LABEL_WIDTH = "115px"
    BOX_WIDTH = "145px"
    CONTROL_WIDTH = "285px"

    def make_row(label_html, widget, fontsize="16px"):
        widget.layout = widgets.Layout(width=BOX_WIDTH)
        label = widgets.HTML(
            value=f"""
            <span style="font-size:{fontsize}; line-height:1.2;">
                {label_html}
            </span>
            """,
            layout=widgets.Layout(width=LABEL_WIDTH),
        )
        return widgets.HBox(
            [label, widget],
            layout=widgets.Layout(
                align_items="center",
                width=CONTROL_WIDTH,
            ),
        )

    def show_error(message):
        display(
            widgets.HTML(
                f"""
                <div style="
                    color:#8a1f11;
                    background:#fff3f0;
                    border:1px solid #f0b4a8;
                    border-radius:6px;
                    padding:10px;
                    font-size:14px;
                    max-width:720px;
                ">
                    <b>Error:</b> {message}
                </div>
                """
            )
        )

    input_dropdown = widgets.Dropdown(
        options=[
            "Exchange current density",
            "Transfer coefficient",
            "Number of electrons",
            "Temperature",
        ],
        value="Transfer coefficient",
        layout=widgets.Layout(width=BOX_WIDTH),
    )
    input_row = make_row("Input:", input_dropdown)

    sweep_text = widgets.Text(
        value="0.3, 0.5, 0.7",
        placeholder="comma-separated values",
    )
    sweep_row = make_row("Sweep:", sweep_text)

    eta_min_box = widgets.FloatText(value=-0.30)
    eta_min_row = make_row("η<sub>min</sub> (V):", eta_min_box)

    eta_max_box = widgets.FloatText(value=0.30)
    eta_max_row = make_row("η<sub>max</sub> (V):", eta_max_box)

    base_i0_box = widgets.FloatText(value=1.0)
    base_i0_row = make_row("base i<sub>0</sub> (A/m²):", base_i0_box)

    base_alpha_box = widgets.FloatText(value=0.5)
    base_alpha_row = make_row("base α:", base_alpha_box)

    base_n_box = widgets.FloatText(value=1.0)
    base_n_row = make_row("base n:", base_n_box)

    base_T_box = widgets.FloatText(value=298.15)
    base_T_row = make_row("base T (K):", base_T_box)

    plot_button = widgets.Button(
        description="Plot",
        button_style="success",
        layout=widgets.Layout(width="120px"),
    )
    plot_row = widgets.HBox(
        [widgets.HTML(layout=widgets.Layout(width=LABEL_WIDTH)), plot_button]
    )

    out = widgets.Output(layout=widgets.Layout(width="auto", border="none"))

    def set_default_sweep(*args):
        name = input_dropdown.value
        if name == "Exchange current density":
            sweep_text.value = "0.1, 1, 10"
        elif name == "Transfer coefficient":
            sweep_text.value = "0.3, 0.5, 0.7"
        elif name == "Number of electrons":
            sweep_text.value = "1, 2, 4"
        elif name == "Temperature":
            sweep_text.value = "273.15, 298.15, 323.15"

    input_dropdown.observe(set_default_sweep, names="value")

    def on_plot_clicked(button):
        out.clear_output(wait=True)
        plt.close("all")

        try:
            sweep_values = parse_sweep_values(sweep_text.value)
            fig = plot_butler_volmer_sweep(
                input_name=input_dropdown.value,
                sweep_values=sweep_values,
                eta_min=eta_min_box.value,
                eta_max=eta_max_box.value,
                npts=500,
                base_i0=base_i0_box.value,
                base_alpha=base_alpha_box.value,
                base_n=base_n_box.value,
                base_T=base_T_box.value,
            )

            with out:
                display(fig)
                plt.close(fig)

        except Exception as exc:
            with out:
                show_error(str(exc))

    plot_button.on_click(on_plot_clicked)

    base_header = widgets.HTML(
        """
        <div style="
            font-size:14px;
            font-weight:600;
            margin-top:8px;
            margin-bottom:4px;
        ">
            Base values used for parameters not being swept:
        </div>
        """
    )


    def update_visible_base_inputs(*args):
        swept = input_dropdown.value

        base_i0_row.layout.display = (
            "" if swept != "Exchange current density" else "none"
        )
        base_alpha_row.layout.display = (
            "" if swept != "Transfer coefficient" else "none"
        )
        base_n_row.layout.display = (
            "" if swept != "Number of electrons" else "none"
        )
        base_T_row.layout.display = (
            "" if swept != "Temperature" else "none"
        )


    try:
        input_dropdown.unobserve(update_visible_base_inputs, names="value")
    except Exception:
        pass

    input_dropdown.observe(update_visible_base_inputs, names="value")
    update_visible_base_inputs()

    controls = widgets.VBox(
        [
            input_row,
            sweep_row,
            eta_min_row,
            eta_max_row,
            base_header,
            base_i0_row,
            base_alpha_row,
            base_n_row,
            base_T_row,
            plot_row,
        ],
        layout=widgets.Layout(
            width=CONTROL_WIDTH,
            min_width=CONTROL_WIDTH,
            margin="0 10px 0 0",
        ),
    )

    output_panel = widgets.Box(
        [out],
        layout=widgets.Layout(
            flex="1 1 auto",
            width="calc(100% - 295px)",
            min_width="720px",
        ),
    )

    ui = widgets.HBox(
        [controls, output_panel],
        layout=widgets.Layout(
            width="100%",
            align_items="flex-start",
            justify_content="flex-start",
        ),
    )

    display(ui)

In [ ]:
from pathlib import Path

figure_dir = Path("exercise4_figures")
figure_dir.mkdir(exist_ok=True)

# Main 5-panel title / overview figure.
fig = plot_butler_volmer_sweep(
    input_name="Transfer coefficient",
    sweep_values=[0.3, 0.5, 0.7],
    eta_min=-0.30,
    eta_max=0.30,
    npts=500,
    base_i0=1.0,
    base_alpha=0.5,
    base_n=1.0,
    base_T=298.15,
    save_path=figure_dir / "exercise4_butler_volmer_title_figure.png",
    dpi=300,
)

# Optional PDF export for slides / LaTeX.
fig.savefig(
    figure_dir / "exercise4_butler_volmer_title_figure.pdf",
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

print("Saved figures to:", figure_dir.resolve())

In [ ]:
from pathlib import Path

figure_dir = Path("exercise4_figures")
figure_dir.mkdir(exist_ok=True)

fig_q1 = plot_question1_bv_decomposition_guide_2x3(
    alpha_values=[0.3, 0.5, 0.7],
    i0_values=[0.1, 1.0, 10.0],
    save_path=figure_dir / "question1_bv_decomposition_2x3.png",
    dpi=300,
)

plt.show()

# Question 2:
# Main Tafel approximation guide: alpha changes the anodic and cathodic slopes.
fig_q2_alpha = plot_question2_tafel_guide_general(
    input_name="Transfer coefficient",
    sweep_values=[0.3, 0.5, 0.7],
    save_path=figure_dir / "question2_tafel_alpha_sweep.png",
    dpi=300,
)

# Optional Question 2 support:
# Number of electrons: increasing n decreases the Tafel slope magnitude.
fig_q2_n = plot_question2_tafel_guide_general(
    input_name="Number of electrons",
    sweep_values=[1, 2, 4],
    save_path=figure_dir / "question2_tafel_n_sweep.png",
    dpi=300,
)

# Optional Question 2 support:
# Temperature: increasing T increases the Tafel slope magnitude.
fig_q2_T = plot_question2_tafel_guide_general(
    input_name="Temperature",
    sweep_values=[273.15, 298.15, 323.15],
    save_path=figure_dir / "question2_tafel_temperature_sweep.png",
    dpi=300,
)

plt.show()

print("Saved figures:")
print(figure_dir / "question1_bv_decomposition_alpha_sweep.png")
print(figure_dir / "question2_tafel_alpha_sweep.png")
print(figure_dir / "question2_tafel_n_sweep.png")
print(figure_dir / "question2_tafel_temperature_sweep.png")


from pathlib import Path

figure_dir = Path("exercise4_figures")
figure_dir.mkdir(exist_ok=True)

fig_q2 = plot_question2_tafel_guides_1x3(
    alpha_values=[0.3, 0.5, 0.7],
    n_values=[1, 2, 4],
    T_values=[273.15, 298.15, 323.15],
    save_path=figure_dir / "question2_tafel_guides_1x3.png",
    dpi=300,
)

plt.show()